<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 9.2**
---

# **Razonamiento**

# **STaR: Self-Taught Reasoner**

La técnica de entrenamiento **STaR** (siglas en inglés de *Self-Taught Reasoner* o "Razonador Autodidacta") es un método innovador que permite a los Modelos de Lenguaje (LLMs) **enseñarse a sí mismos a razonar** y mejorar su lógica sin depender masivamente de datos etiquetados por humanos.

En términos sencillos, funciona como un estudiante que hace ejercicios: primero intenta resolverlos, comprueba las soluciones y, si se equivocó, mira la respuesta correcta e intenta deducir **por qué** esa es la solución para aprender del error.

<div style="background-color: #e4f0f8ff; border-left: 5px solid #ffffffff; padding: 1.5em; margin: 30px; width: 600px">
El artículo científico que introdujo formalmente el método STaR se titula: <b>STaR: Bootstrapping Reasoning With Reasoning</b>. Fue publicado originalmente en 2022 y presentado en la prestigiosa conferencia NeurIPS (Neural Information Processing Systems) de ese mismo año. <a href="https://arxiv.org/pdf/2203.14465" target="_blank">Puedes leer el artículo completo aquí.</a>
</div>

### **¿Cómo funciona el ciclo STaR?**

La técnica se basa en un bucle de mejora continua (*bootstrapping*) que sigue estos pasos principales:

1.  **Generación de Razonamiento:**
    Al modelo se le presentan muchas preguntas. Se le pide que genere una respuesta paso a paso (lo que se conoce como *Chain of Thought* o cadena de pensamiento) para intentar llegar a la solución.

2.  **Filtrado (Validación):**
    El sistema comprueba si la respuesta final a la que llegó el modelo es correcta.
    * **Si acertó:** Ese razonamiento se guarda como un "ejemplo de éxito" para entrenar al modelo más tarde.
    * **Si falló:** Se descarta el razonamiento erróneo, pero comienza una fase clave llamada "racionalización".
    
    ¿Cómo sabe el sistema si la respuesta es correcta? Normalmente, se utiliza un conjunto de datos con respuestas conocidas (etiquetadas) para validar las respuestas del modelo. Los ejemplos con los que trabaja el modelo suelen provenir de conjuntos de datos estándar en tareas específicas (como matemáticas, lógica, etc.) donde las respuestas correctas ya están definidas.

3.  **Racionalización (Aprender del error):**
    Esta es la gran innovación de STaR. Si el modelo falló, el sistema le da la respuesta correcta y le dice: *"La solución es X, genera un nuevo razonamiento que explique por qué esta es la respuesta correcta"*.
    Si el modelo logra construir una lógica válida que llegue a esa respuesta correcta, este nuevo dato también se guarda como material de entrenamiento.

4.  **Reentrenamiento (Fine-tuning):**
    Finalmente, se reentrena al modelo utilizando todos los razonamientos exitosos (tanto los que acertó a la primera como los que corrigió con ayuda). Esto mejora su capacidad para la siguiente ronda. Este reentrenamiento se realiza utilizando *supervised fine-tuning* (SFT), donde el modelo aprende de los ejemplos exitosos generados en las etapas anteriores.

<div align="center">
  <img src="imgs/STaR.jpg" width="600px">
</div>


### **¿Por qué es diferente a otros métodos?**

Lo revolucionario de STaR es que rompe con dos limitaciones clásicas del entrenamiento de IA:

* **Evita el "aprender de memoria":** En lugar de entrenar al modelo solo para predecir la respuesta final (ej. "La respuesta es 5"), se le entrena para producir el **proceso lógico** que lleva al 5.

* **Aprovecha los fallos:** La mayoría de los entrenamientos descartan los intentos fallidos. STaR los recicla dándole al modelo una "pista" (la respuesta correcta) para que aprenda a justificarla, convirtiendo un error en un dato de entrenamiento útil.

### **Principales Ventajas**

* **Autonomía:** Reduce la necesidad de tener miles de explicaciones escritas por humanos costosas de obtener.

* **Mejora en Matemáticas y Lógica:** Ha demostrado ser excepcionalmente útil en problemas complejos donde la intuición no basta y se requiere una secuencia de pasos (como matemáticas o sentido común).

* **Escalabilidad:** El modelo puede volverse cada vez más inteligente iterando este proceso varias veces, resolviendo problemas cada vez más difíciles que él mismo ayudó a desglosar en iteraciones anteriores.

### **Controversias y Debates sobre STaR**

Aunque el método STaR ha demostrado ser muy eficaz (y es la base de modelos modernos de razonamiento como o1 de OpenAI o DeepSeek-R1), genera cierta controversia y debate en la comunidad científica por varias limitaciones fundamentales y riesgos teóricos.

#### **1. El problema de "Acertar por las razones equivocadas" (False Positives)**
Esta es la crítica técnica más fuerte. Como el filtro de la Fase 2 solo verifica la **respuesta final** y no el proceso, el sistema es ciego a la lógica defectuosa.

* **El escenario:** Imagina que la pregunta es compleja. El modelo alucina una lógica absurda pero, por pura suerte o correlación espuria, llega al número correcto.

* **El riesgo:** STaR etiqueta ese razonamiento basura como "correcto" y entrena al modelo con él. Esto puede reforzar **alucinaciones lógicas**, enseñando al modelo que inventar pasos falsos es válido siempre que el resultado coincida.


#### **2. La artificialidad de la "Racionalización" (Sesgo Retrospectivo)**

La fase en la que se le da la respuesta correcta al modelo y se le pide que justifique *"por qué es así"* es muy controvertida.

* **Crítica:** Esto no es razonamiento real, es **ingeniería inversa**. Al forzar al modelo a conectar una pregunta con una respuesta predeterminada, se le puede estar enseñando a ser un "retórico complaciente" que inventa justificaciones persuasivas (aunque sean falsas) para defender un hecho, en lugar de deducir la verdad.

* **Efecto secundario:** El modelo aprende a ser muy convincente incluso cuando se equivoca.


#### **3. Dependencia de la "Verdad Objetiva" (Ground Truth)**

STaR solo funciona bien en dominios cerrados (Matemáticas, Código, Lógica formal).

* **Limitación:** No se puede aplicar fácilmente a tareas subjetivas, creativas o matizadas (como ética, política 
o escritura) porque no existe una "hoja de respuestas" automática para filtrar el éxito.

* **Controversia:** Los críticos argumentan que esto limita enormemente su utilidad para la Inteligencia General Artificial (AGI), ya que el mundo real rara vez tiene respuestas binarias claras.


#### **4. Riesgo de "Colapso del Modelo" (Echo Chambers)**
Al entrenarse con sus propios datos sintéticos, existe el riesgo de que el modelo pierda diversidad y creatividad.
Si el modelo empieza a preferir una forma específica de razonar (aunque no sea la mejor), el ciclo de reentrenamiento amplificará esa tendencia hasta que el modelo se vuelva rígido y dogmático en su forma de resolver problemas, perdiendo la capacidad de encontrar soluciones alternativas.


#### **5. ¿Es realmente "razonamiento"?**
Existe un debate filosófico y técnico sobre si STaR enseña a razonar o simplemente a **imitar la estructura del razonamiento**. Algunos investigadores sugieren que el modelo simplemente aprende patrones estadísticos más largos (cadenas de texto que *parecen* explicaciones) para llegar al token final, sin entender realmente la relación causal entre los pasos.


# **DeepSeek-R1-Zero**

**DeepSeek-R1 Zero** es un modelo experimental que marcó un hito en la Inteligencia Artificial reciente (publicado a principios de 2025) porque demostró una hipótesis audaz: **el razonamiento complejo puede surgir puramente del Aprendizaje por Refuerzo (RL), sin necesidad de enseñar al modelo cómo pensar primero.**

Si STaR es como un estudiante que estudia de sus propios apuntes corregidos, **DeepSeek-R1 Zero** es como un estudiante que nunca ha visto un libro de texto, pero que se le da un examen tras otro y se le dice únicamente "Correcto" o "Incorrecto", hasta que él solo inventa las reglas de la lógica para aprobar.

### **1. La premisa "Zero": Sin SFT Inicial**
Lo que hace único a este modelo (y de donde viene el nombre "Zero") es que **se saltó la fase de Supervised Fine-Tuning (SFT)**.

* **Enfoque tradicional (y STaR):** Primero tomas un modelo base y le das miles de ejemplos de "Pregunta $\rightarrow$ Razonamiento paso a paso $\rightarrow$ Respuesta" escritos por humanos o curados (SFT). Esto le enseña al modelo el *formato* de pensar. Luego aplicas RL para pulirlo.

* **Enfoque DeepSeek-R1 Zero:** Tomaron el modelo base (DeepSeek-V3-Base) y le aplicaron **directamente** un algoritmo de RL a gran escala sin enseñarle ni un solo ejemplo de cómo razonar.


### **2. El motor: GRPO (Group Relative Policy Optimization)**

Para lograr esto sin que el costo computacional fuera astronómico, no usaron el algoritmo clásico PPO (que requiere mucha memoria). Usaron **GRPO**.

* **Cómo funciona:**
    1.  Le dan una pregunta al modelo.
    2.  El modelo genera un grupo de salidas distintas (por ejemplo, 6 intentos diferentes).
    3.  Se evalúa el grupo: ¿Cuáles llegaron a la respuesta correcta?
    4.  Se optimiza el modelo premiando las que funcionaron mejor *en relación* con el promedio del grupo.

* **La recompensa:** Fue extremadamente simple. Solo se premiaba la **exactitud** (¿llegaste al resultado?) y un poco el **formato** (¿usaste las etiquetas `<think>` y `<answer>` correctamente?). No se le premió por "razonar bonito".

<br>
<br>
<div align="center">
  <img src="imgs/DeepSeek-R1-Zero.jpg" width="500px">
</div>
<br>
    

### **3. Comportamiento Emergente**
Lo más fascinante de DeepSeek-R1 Zero es que, tras miles de pasos de entrenamiento con RL, el modelo **desarrolló espontáneamente** capacidades de razonamiento (Chain of Thought) que nadie le había enseñado.

Los investigadores observaron que el modelo aprendió por sí mismo a:

* **Autocorregirse:** En sus textos generados aparecían frases como *"Espera, esto no está bien, voy a intentar otro enfoque..."*.

* **Verificar:** *"Voy a volver a calcular esto para asegurarme"*.

* **Dedicar más tiempo:** Aprendió que pensar durante más tiempo (generar más tokens de pensamiento) aumentaba su recompensa.



### **4. Comparación: STaR vs. DeepSeek-R1 Zero**
Es importante diferenciarlo de la técnica STaR que vimos antes:

| Característica | STaR (Self-Taught Reasoner) | DeepSeek-R1 Zero |
| :--- | :--- | :--- |
| **Entrenamiento** | **Iterativo SFT:** Genera datos, filtra y reentrena como si fuera aprendizaje supervisado. | **Puro RL:** Actualiza los pesos basándose directamente en la recompensa (gradientes), sin crear "datasets" intermedios. |
| **Origen del razonamiento** | El modelo imita razonamientos previos que fueron exitosos. | El modelo **inventa** el razonamiento para maximizar la recompensa. |
| **Dependencia** | Requiere un modelo que ya sepa razonar un poco (o un prompt muy bueno) para arrancar. | Puede arrancar desde un modelo base "crudo" (Cold Start). |

### **5. ¿Por qué no usamos "Zero" en producción? (Sus limitaciones)**
Aunque R1-Zero fue un éxito científico, tenía problemas de usabilidad que obligaron a crear la versión final (DeepSeek-R1):

* **Ilegibilidad:** Como nadie le enseñó a escribir para humanos, su "pensamiento" era caótico, difícil de leer y mezclaba idiomas sin control.

* **Pobres habilidades sociales:** Al optimizar solo para la respuesta correcta, el modelo podía ser brusco o ignorar instrucciones de formato (ej. "responde en forma de poema").

* **Bucle infinito:** A veces entraba en bucles de pensamiento de los que no salía.


# **DeepSeek-R1**

**DeepSeek-R1** es un Modelo de Lenguaje Grande (LLM) de código abierto (open weights) especializado en razonamiento complejo (matemáticas, programación, lógica). Logró un rendimiento comparable al modelo **o1 de OpenAI**, pero siendo de código abierto y con un coste de entrenamiento drásticamente menor. La diferencia con **R1-Zero** es que, mientras que *Zero* era puro RL y, a veces caótico o ilegible, **R1** es legible, coherente y fácil de usar, manteniendo la potencia de razonamiento.


### **1. Pipeline de Entrenamiento**

A diferencia de "Zero", R1 no usa solo RL desde el principio. Utiliza un proceso híbrido de 4 fases para garantizar calidad y estabilidad:

#### **Fase 1: Cold Start (Arranque en Frío)**
* **Problema de Zero:** Al empezar solo con RL, el modelo tardaba mucho en converger y escribía mal.
* **Solución:** Se crea un pequeño dataset de alta calidad con razonamientos largos (Chain of Thought - CoT) escritos o curados por humanos/modelos previos.
* **Acción:** Se hace un **Fine-Tuning Supervisado (SFT)** inicial con estos pocos datos.
* **Resultado:** El modelo aprende *cómo* estructurar su pensamiento (legibilidad) antes de empezar a aprender *qué* pensar.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_01.jpg" width="500px">
</div>
<br>

#### **Fase 2: Reasoning RL (Refuerzo para Razonar)**
* Aquí se aplica la técnica vista en R1-Zero (**GRPO**).
* El modelo practica masivamente problemas de matemáticas y código.
* Se premia la exactitud de la respuesta.
* **Resultado:** El modelo se vuelve extremadamente inteligente y capaz de corregirse a sí mismo.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_02.jpg" width="500px">
</div>
<br>

#### **Fase 3: Rejection Sampling & SFT (Generalización)**
* **El paso clave:** Usando el modelo de la Fase 2, se generan miles de respuestas y razonamientos.
* **Filtrado:** Se guardan solo las respuestas correctas y legibles (similar a **STaR**).
* **Ampliación:** Se añaden datos que *no* son de razonamiento (escritura creativa, traducción, conocimientos generales) para que el modelo no sea solo una calculadora, sino un asistente útil.
* **Acción:** Se reentrena el modelo (SFT) con este dataset masivo y limpio.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_03.jpg" width="500px">
</div>
<br>

Para entender cómo se construye el modelo final DeepSeek-R1 en esta tercera fase, debemos visualizar dos corrientes paralelas de generación de datos que funcionan con filosofías opuestas, pero que se complementan para crear el resultado final.

Por un lado tenemos la rama del razonamiento (la columna izquierda de nuestro diagrama mental). Aquí, el protagonista es el modelo "genio pero caótico" proveniente de la fase anterior, el DeepSeek-V3-2. Su misión es generar soluciones a problemas complejos de matemáticas y programación. Lo crucial en este lado es que el juez que valida los datos es una herramienta externa totalmente objetiva y determinista, como un compilador de código o una calculadora. Dado que buscamos la verdad científica, el filtro es implacable: si el resultado numérico no es exacto o el código no funciona, el dato se descarta inmediatamente. De este proceso de "Rejection Sampling" extraemos unas 600.000 muestras de pura inteligencia lógica verificada.

Por otro lado corre la rama general (la columna derecha), cuyo objetivo es muy distinto: asegurar que el modelo no pierda su humanidad ni sus habilidades lingüísticas. Aquí no utilizamos al modelo experimental, sino al DeepSeek-V3-Base estándar o datos humanos, ya que escriben mejor y son más educados. Se generan contenidos subjetivos como poesía, traducciones o respuestas de cultura general. A diferencia de la rama anterior, aquí no existe una fórmula matemática para validar si un poema es "correcto". Por tanto, el juicio es más suave y probabilístico; se confía en la capacidad del modelo base y se utilizan modelos de recompensa que simplemente verifican que el estilo sea fluido, útil y seguro. Estas 200.000 muestras no buscan descubrir nueva lógica, sino mantener la capacidad de conversar.

Finalmente, estas dos corrientes desembocan en un mismo punto. La rigurosidad verificada de la izquierda y la fluidez comunicativa de la derecha se mezclan en un único dataset de 800.000 ejemplos. Este *dataset* definitivo se utiliza para entrenar (SFT) desde cero a una copia limpia del modelo base, transfiriéndole así la capacidad de razonar profundamente sin que herede los vicios ni el caos del proceso de aprendizaje original.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_04.jpg" width="500px">
</div>
<br>

#### **Fase 4: RL for All Scenarios (Alineación Final)**
* Se aplica una ronda final de RL.
* **Objetivo:** Alinear el modelo con valores humanos (seguridad, no ser dañino) y pulir la utilidad en tareas generales.
* Se usan recompensas tanto por la respuesta final como por la seguridad del proceso.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_05.jpg" width="500px">
</div>
<br>


## **Destilación de Modelos (Distillation)**
DeepSeek no solo liberó el modelo gigante (671B parámetros). Se usaron los razonamientos generados por R1 para entrenar modelos pequeños y densos (basados en Llama y Qwen) de 1.5B, 7B, 8B, 14B, 32B y 70B. Con esto demostraron que los modelos pequeños pueden razonar increíblemente bien si se les entrena con los "pensamientos" de un modelo gigante.

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_06.jpg" width="500px">
</div>
<br>

<br>
<div align="center">
  <img src="imgs/DeepSeek-R1_07.jpg" width="500px">
</div>
<br>

### **Comparativa con OpenAI o1**

| Característica | DeepSeek-R1 | OpenAI o1 |
| :--- | :--- | :--- |
| **Acceso** | Open Source (Pesos libres) | Propietario (API cerrada) |
| **Rendimiento** | Empate técnico en Math (AIME) y Code (Codeforces) | Muy alto |
| **Transparencia** | Puedes ver el `<think>` (proceso de pensamiento) | El pensamiento está oculto |
| **Coste** | Entrenamiento muy barato (relativo a la industria) | Entrenamiento extremadamente costoso |

